In [1]:
!nvidia-smi

Sun Apr  5 20:57:06 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   32C    P0             43W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
import torch
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

Torch version: 2.10.0+cu128
CUDA available: True
GPU name: NVIDIA A100-SXM4-40GB


In [5]:
import os, getpass
token = getpass.getpass("GitHub token: ")
!git clone https://{token}@github.com/jasmineztruong8/efficient-codegen.git
%cd efficient-codegen

GitHub token: ··········
Cloning into 'efficient-codegen'...
remote: Enumerating objects: 157, done.
remote: Counting objects: 100% (17/17), done.
remote: Compressing objects: 100% (15/15), done.
remote: Total 157 (delta 4), reused 11 (delta 2), pack-reused 140 (from 1)
Receiving objects: 100% (157/157), 15.67 MiB | 21.98 MiB/s, done.
Resolving deltas: 100% (62/62), done.
/content/efficient-codegen


In [7]:
!git checkout jg/training-data-construction
!git log --oneline

Already on 'jg/training-data-construction'
Your branch is up to date with 'origin/jg/training-data-construction'.
9c80655 (HEAD -> jg/training-data-construction, origin/jg/training-data-construction) Generate runtime aware and control datasets from benchmarked candidates
1c5237e (origin/model-skeleton) Merge pull request #3 from jasmineztruong8/profiling-pipeline
924867a (origin/profiling-pipeline) Add generated candidates full dataset and benchmarked full candidates dataset
f413100 Merge pull request #2 from jasmineztruong8/profiling-pipeline
0f2cb50 Removed unnecessary cell outputs and updated README
93ccfe0 Merge pull request #1 from jasmineztruong8/profiling-pipeline
61e906a Add profile_operators.py
149c11c implement training for baseline (control model) and efficient codegen sft model
2175be2 adding trace-level profile operators
5202583 Add timeout protection and Pass@1 stats to evaluate_candidates.py
4a3959a Add GPU memory monitoring to optimize generation wtih batching
5f8ef88 A

In [8]:
!ls training/data
!wc -l training/data/control.jsonl training/data/runtime_aware.jsonl

control.jsonl  runtime_aware.jsonl
   2110 training/data/control.jsonl
   2110 training/data/runtime_aware.jsonl
   4220 total


In [9]:
!pip install -q transformers datasets accelerate peft bitsandbytes wandb trl torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 45.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 630.8/630.8 kB 54.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 50.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 56.7 MB/s eta 0:00:00


In [3]:
%cd /content/efficient-codegen
!ls training/data

/content/efficient-codegen
control.jsonl  runtime_aware.jsonl


In [9]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [10]:
!mkdir -p /content/drive/MyDrive/efficient-codegen/checkpoints

In [11]:
!python training/train.py \
  --mode control \
  --data_path training/data/control.jsonl \
  --output_dir /content/drive/MyDrive/efficient-codegen/checkpoints/control_smoke \
  --limit 100 \
  --num_train_epochs 1 \
  --per_device_train_batch_size 2 \
  --gradient_accumulation_steps 2

Loading weights: 100% 338/338 [00:01<00:00, 274.43it/s, Materializing param=model.norm.weight]
Training on 100 examples  (mode=control)
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
Adding EOS to train dataset: 100% 100/100 [00:00<00:00, 19353.56 examples/s]
Tokenizing train dataset: 100% 100/100 [00:00<00:00, 1153.98 examples/s]
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.
{'loss': '1.103', 'grad_norm': '0.7617', 'learning_rate': '0.0001577', 'entropy': '0.9053', 'num_tokens': '9054', 'mean_token_accuracy': '0.7723', 'epoch': '0.4'}
{'loss': '0.5991', 'grad_norm': '0.8828', 'learning_rate': '3.174e-05', 'entropy': '0.6101', 'num_tokens': '1.791e+04', 'mean_token_accuracy': '0.8542', 'epoch': '0.8'}
{'train_runtime': '29.64', '

In [12]:
!python training/train.py \
  --mode runtime_aware \
  --data_path training/data/runtime_aware.jsonl \
  --output_dir /content/drive/MyDrive/efficient-codegen/checkpoints/runtime_aware_smoke \
  --limit 100 \
  --num_train_epochs 1 \
  --per_device_train_batch_size 2 \
  --gradient_accumulation_steps 2

Loading weights: 100% 338/338 [00:01<00:00, 293.36it/s, Materializing param=model.norm.weight]
Training on 100 examples  (mode=runtime_aware)
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
Adding EOS to train dataset: 100% 100/100 [00:00<00:00, 18479.55 examples/s]
Tokenizing train dataset: 100% 100/100 [00:00<00:00, 1168.34 examples/s]
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.
{'loss': '1.121', 'grad_norm': '0.957', 'learning_rate': '0.0001577', 'entropy': '0.9205', 'num_tokens': '8829', 'mean_token_accuracy': '0.7703', 'epoch': '0.4'}
{'loss': '0.6087', 'grad_norm': '0.8594', 'learning_rate': '3.174e-05', 'entropy': '0.6222', 'num_tokens': '1.749e+04', 'mean_token_accuracy': '0.8507', 'epoch': '0.8'}
{'train_runtime': '29.8

In [13]:
!ls /content/drive/MyDrive/efficient-codegen/checkpoints/control_smoke
!ls /content/drive/MyDrive/efficient-codegen/checkpoints/runtime_aware_smoke

adapter_config.json	   checkpoint-25	  tokenizer.json
adapter_model.safetensors  README.md		  training_args.bin
chat_template.jinja	   tokenizer_config.json
adapter_config.json	   checkpoint-25	  tokenizer.json
adapter_model.safetensors  README.md		  training_args.bin
chat_template.jinja	   tokenizer_config.json
